In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

from pyspark.sql.functions import col, row_number, to_date, round, year, month, dayofmonth, quarter, dayofweek, when, lpad, concat_ws
from pyspark.sql.window import Window

StatementMeta(, d01f92bf-ded4-439c-86c4-5b1805e31b1a, 3, Finished, Available, Finished, False)

In [2]:
# Leitura da Camada Raw

tabela_raw = "base_vendas"
df_raw = spark.read.table(tabela_raw)

StatementMeta(, d01f92bf-ded4-439c-86c4-5b1805e31b1a, 4, Finished, Available, Finished, False)

In [3]:
# Criação de dimensões
# Seleciona as colunas específicas e remove as duplicatas
# Define a regra de ordenação para gerar o ID sequencial
# Cria a nova coluna 'sk'

# --- 1. DIM_VENDEDOR ---
df_vendedor_distinto = df_raw.select("vendedor_cod", "vendedor_nom", "vendedor_supervisor", "vendedor_equipe").distinct()
window_vendedor = Window.orderBy("vendedor_cod")
df_dim_vendedor = df_vendedor_distinto.withColumn("sk_vendedor", row_number().over(window_vendedor))

# --- 2. DIM_CLIENTE ---
df_cliente_distinto = df_raw.select("cliente_cod", "cliente_nome", "cliente_segmento", "cidade_cod", "cidade_nom").distinct()
window_cliente = Window.orderBy("cliente_cod")
df_dim_cliente = df_cliente_distinto.withColumn("sk_cliente", row_number().over(window_cliente))

# --- 3. DIM_PRODUTO ---
df_produto_distinto = df_raw.select("produto_cod", "produto_nome", "produto_grupo", "produto_linha", "produto_fornecedor", "produto_custo_unitario").distinct()
window_produto = Window.orderBy("produto_cod")
df_dim_produto = df_produto_distinto.withColumn("sk_produto", row_number().over(window_produto))

StatementMeta(, d01f92bf-ded4-439c-86c4-5b1805e31b1a, 5, Finished, Available, Finished, False)

In [5]:
# Criação da tabela Fato Vendas

df_fat_venda = df_raw \
    .join(df_dim_vendedor, on="vendedor_cod", how="left") \
    .join(df_dim_cliente, on="cliente_cod", how="left") \
    .join(df_dim_produto, on="produto_cod", how="left") \
    .withColumn("valor_total_item", round(col("qtd_itens") * col("preco_unitario"), 2)) \
    .select(
        col("sk_vendedor"),
        col("sk_cliente"),
        col("sk_produto"),
        col("num_venda"),
        col("data_venda"),
        col("data_entrega_prevista"),
        col("data_entrega"),
        col("qtd_itens"),
        col("preco_unitario"),
        col("valor_bruto"),
        col("valor_total_item"),
        col("ocorrencia_devolucao")
    )

StatementMeta(, d01f92bf-ded4-439c-86c4-5b1805e31b1a, 7, Finished, Available, Finished, False)

In [15]:
# Salvando as tabelas na camada Curated (Formato Delta)


# Caminho físico raiz da pasta "Tables" no Lakehouse Curated.
caminho_curated = "abfss://DataSide_Analytics@onelake.dfs.fabric.microsoft.com/LH_Vendas_Curated.Lakehouse/Tables"

# Salvando DIM_VENDEDOR
df_dim_vendedor.select("sk_vendedor", "vendedor_cod", "vendedor_nom", "vendedor_supervisor", "vendedor_equipe") \
    .write.format("delta").mode("overwrite").save(f"{caminho_curated}/dim_vendedor")

# Salvando DIM_CLIENTE
df_dim_cliente.select("sk_cliente", "cliente_cod", "cliente_nome", "cliente_segmento", "cidade_cod", "cidade_nom") \
    .write.format("delta").mode("overwrite").save(f"{caminho_curated}/dim_cliente")

# Salvando DIM_PRODUTO
df_dim_produto.select("sk_produto", "produto_cod", "produto_nome", "produto_grupo", "produto_linha", "produto_fornecedor", "produto_custo_unitario") \
    .write.format("delta").mode("overwrite").save(f"{caminho_curated}/dim_produto")

# Salvando FAT_VENDA
df_fat_venda.write.format("delta").mode("overwrite").save(f"{caminho_curated}/fat_venda")

StatementMeta(, d01f92bf-ded4-439c-86c4-5b1805e31b1a, 17, Finished, Available, Finished, False)

Código executado! As tabelas do Modelo Dimensional foram processadas e salvas diretamente no Lakehouse Curated via OneLake!


In [8]:
# Gerando tabela Dim_Calendario pronta para analise

# Parâmetros de data ajustados para cobrir a base de 2019 até 2025
data_inicio = '2019-01-01'
data_fim = '2025-12-31'

df_calendario_base = spark.sql(f"SELECT explode(sequence(to_date('{data_inicio}'), to_date('{data_fim}'), interval 1 day)) as data")

# Criamos as colunas de inteligência de tempo
df_dim_calendario = df_calendario_base \
    .withColumn("ano", year("data")) \
    .withColumn("mes", month("data")) \
    .withColumn("dia", dayofmonth("data")) \
    .withColumn("trimestre", quarter("data")) \
    .withColumn("dia_semana", dayofweek("data")) \
    .withColumn("ano_mes_num", concat_ws("-", col("ano"), lpad(col("mes"), 2, "0"))) # Ex: 2019-01

# Traduzindo os nomes dos meses para Português
df_dim_calendario = df_dim_calendario.withColumn(
    "nome_mes",
    when(col("mes") == 1, "Janeiro")
    .when(col("mes") == 2, "Fevereiro")
    .when(col("mes") == 3, "Março")
    .when(col("mes") == 4, "Abril")
    .when(col("mes") == 5, "Maio")
    .when(col("mes") == 6, "Junho")
    .when(col("mes") == 7, "Julho")
    .when(col("mes") == 8, "Agosto")
    .when(col("mes") == 9, "Setembro")
    .when(col("mes") == 10, "Outubro")
    .when(col("mes") == 11, "Novembro")
    .otherwise("Dezembro")
)

# 5. Traduzindo os dias da semana para Português (no Spark, domingo = 1)
df_dim_calendario = df_dim_calendario.withColumn(
    "nome_dia_semana",
    when(col("dia_semana") == 1, "Domingo")
    .when(col("dia_semana") == 2, "Segunda-feira")
    .when(col("dia_semana") == 3, "Terça-feira")
    .when(col("dia_semana") == 4, "Quarta-feira")
    .when(col("dia_semana") == 5, "Quinta-feira")
    .when(col("dia_semana") == 6, "Sexta-feira")
    .otherwise("Sábado")
)

StatementMeta(, d01f92bf-ded4-439c-86c4-5b1805e31b1a, 10, Finished, Available, Finished, False)

In [16]:
# Salvando o Calendario na camada Curated (Formato Delta)

# Salvando DIM_CALENDARIO usando o .save()
df_dim_calendario.write.format("delta").mode("overwrite").save(f"{caminho_curated}/dim_calendario")

print("Dimensão Calendário ajustada (2019-2025) e salva com sucesso!")

StatementMeta(, d01f92bf-ded4-439c-86c4-5b1805e31b1a, 18, Finished, Available, Finished, False)

Dimensão Calendário ajustada (2019-2025) e salva com sucesso!
